# 03 · Construcción de la tabla maestra única

Notebook final de la fase de procesamiento. Concatena los 36 dataframes maestros
de `data/master/` (30 temporadas cerradas + 6 snapshots de la 25/26) en una única
tabla que servirá de input para todas las fases posteriores (EDA, FE, ML).

**Operaciones realizadas:**
1. Carga de los 36 archivos `master_<liga>_<temporada>.csv` y `master_<liga>_2526_snapshot_<fecha>.csv`
2. Renombrado de `data_country` → `country` y `data_season` → `season`
3. Gestión de la columna `download_date` (solo presente en snapshots)
4. Concatenación en un único DataFrame
5. Validaciones finales (totales por liga, temporada, matching)
6. Guardado como `master_total.csv` en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
from pathlib import Path

ROOT       = Path.cwd().parents[1]
MASTER_DIR = ROOT / 'data' / 'master'

print('✅ Rutas configuradas')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Master: d:\USER\Desktop\TFM\data\master


## 2. Identificar los archivos maestros

Se localizan los 36 archivos: 30 cerrados (`master_<liga>_<temporada>.csv`) y
6 snapshots (`master_<liga>_2526_snapshot_<fecha>.csv`). Se excluye explícitamente
el propio output `master_total.csv` para evitar leerlo si ya existe.

In [2]:
archivos = sorted([f for f in MASTER_DIR.glob('master_*.csv') if f.name != 'master_total.csv'])

cerrados   = [f for f in archivos if 'snapshot' not in f.name]
snapshots  = [f for f in archivos if 'snapshot' in f.name]

print(f'Total archivos: {len(archivos)}')
print(f'   Cerrados:  {len(cerrados)}')
print(f'   Snapshots: {len(snapshots)}')

if len(cerrados) != 30 or len(snapshots) != 6:
    print('⚠️  Cantidad inesperada de archivos. Revisar.')
else:
    print('✅ 30 cerrados + 6 snapshots = 36 archivos')

Total archivos: 36
   Cerrados:  30
   Snapshots: 6
✅ 30 cerrados + 6 snapshots = 36 archivos


## 3. Carga y concatenación

Para cada archivo se carga el CSV, se asegura que las columnas `data_country`
y `data_season` están presentes, y se concatenan todos en un único DataFrame.
`pd.concat` con `ignore_index=True` reindexa las filas y rellena con `NaN` las
columnas que no existen en algunos archivos (caso de `download_date`, que solo
está en los snapshots).

In [3]:
dfs = []
filas_por_archivo = []

for archivo in archivos:
    df = pd.read_csv(archivo)
    dfs.append(df)
    filas_por_archivo.append({
        'archivo': archivo.name,
        'filas':   len(df),
        'columnas': df.shape[1]
    })

df_total = pd.concat(dfs, ignore_index=True)

print(f'✅ Concatenación completada')
print(f'   Filas totales:    {len(df_total):,}')
print(f'   Columnas totales: {df_total.shape[1]}')

✅ Concatenación completada
   Filas totales:    20,316
   Columnas totales: 122


## 4. Renombrado de columnas de trazabilidad

Las columnas `data_country` y `data_season`, que se vienen arrastrando desde la
ingesta como metadatos, se renombran a `country` y `season` para la tabla final.

Adicionalmente, `season` se mantiene como string (formato `'2425'`, `'2526'`...)
para evitar que pandas la interprete como entero al recargar el CSV en fases posteriores.

In [4]:
df_total = df_total.rename(columns={
    'data_country': 'country',
    'data_season':  'season'
})

df_total['season'] = df_total['season'].astype(str)

print('✅ Columnas renombradas')
print(f"   country: {df_total['country'].unique().tolist()}")
print(f"   season:  {sorted(df_total['season'].unique().tolist())}")

✅ Columnas renombradas
   country: ['england', 'france', 'germany', 'italy', 'spain', 'turkey']
   season:  ['2021', '2122', '2223', '2324', '2425', '2526']


## 5. Gestión de la columna `download_date`

`download_date` solo está presente en las 6 ligas de la 25/26 (los snapshots).
En las 30 temporadas cerradas pandas la rellena automáticamente con `NaN` durante
el `concat`. Esto es exactamente el comportamiento deseado: los `NaN` indican
"dato definitivo" y los valores rellenos indican "snapshot provisional con fecha X".

Cuando se reejecute la fase 03 al cierre de las ligas (mayo/junio), los snapshots
se sustituirán por los masters definitivos sin sufijo, y `download_date` quedará
en `NaN` para todas las filas.

In [5]:
dd_filled = df_total['download_date'].notna().sum()
dd_nan    = df_total['download_date'].isna().sum()

print(f"download_date relleno (snapshots): {dd_filled:,} filas")
print(f"download_date NaN (cerradas):      {dd_nan:,} filas")
print()
print(f"Valores únicos de download_date (excluyendo NaN):")
print(f"   {df_total['download_date'].dropna().unique().tolist()}")

download_date relleno (snapshots): 3,242 filas
download_date NaN (cerradas):      17,074 filas

Valores únicos de download_date (excluyendo NaN):
   [20260428.0]


## 6. Validaciones finales

### 6.1 Distribución de filas por liga y temporada

In [6]:
tabla_distribucion = df_total.groupby(['country', 'season']).size().unstack(fill_value=0)
tabla_distribucion['TOTAL'] = tabla_distribucion.sum(axis=1)
tabla_distribucion.loc['TOTAL'] = tabla_distribucion.sum(axis=0)

tabla_distribucion

season,2021,2122,2223,2324,2425,2526,TOTAL
country,,,,,,,
england,527,538,554,570,562,525,3276
france,573,591,586,525,542,537,3354
germany,496,511,506,494,481,494,2982
italy,602,609,577,590,599,561,3538
spain,571,604,584,598,589,578,3524
turkey,660,635,588,632,580,547,3642
TOTAL,3429,3488,3395,3409,3353,3242,20316


### 6.2 Cobertura de salarios por liga y temporada

In [7]:
cobertura = (
    df_total
    .groupby(['country', 'season'])
    .agg(
        total       = ('player', 'size'),
        con_salario = ('gross_annual_eur', lambda s: s.notna().sum()),
    )
)
cobertura['pct'] = (cobertura['con_salario'] / cobertura['total'] * 100).round(1)
cobertura.unstack(level='season')[['pct']]

pct                              
season   2021  2122  2223  2324  2425  2526
country                                    
england  93.9  92.6  93.7  91.1  95.7  93.7
france   87.4  89.7  89.9  89.3  87.6  83.8
germany  94.8  95.1  95.1  93.9  97.3  92.5
italy    93.9  92.1  96.0  94.1  94.3  91.1
spain    87.4  88.7  87.5  87.5  87.8  85.3
turkey   95.6  88.7  92.0  95.4  91.9  82.6

### 6.3 Resumen global

In [8]:
total       = len(df_total)
con_salario = df_total['gross_annual_eur'].notna().sum()
sin_salario = df_total['gross_annual_eur'].isna().sum()

print(f'Filas totales:           {total:,}')
print(f'Con salario:             {con_salario:,} ({con_salario/total:.1%})')
print(f'Sin salario:             {sin_salario:,} ({sin_salario/total:.1%})')
print(f'Columnas totales:        {df_total.shape[1]}')
print(f'Países distintos:        {df_total["country"].nunique()}')
print(f'Temporadas distintas:    {df_total["season"].nunique()}')
print(f'Equipos distintos:       {df_total["team"].nunique()}')
print(f'Jugadores únicos (id):   {df_total["player_id"].nunique() if "player_id" in df_total.columns else "N/A"}')

Filas totales:           20,316
Con salario:             18,552 (91.3%)
Sin salario:             1,764 (8.7%)
Columnas totales:        122
Países distintos:        6
Temporadas distintas:    6
Equipos distintos:       171
Jugadores únicos (id):   7759


## 7. Guardado

Se guarda la tabla maestra única en `data/master/master_total.csv`.
Este archivo es el input de todas las fases posteriores: EDA, feature engineering,
modelado y visualización.

In [9]:
nombre_salida = 'master_total.csv'
df_total.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Ruta: {MASTER_DIR / nombre_salida}')
print(f'   Filas:    {len(df_total):,}')
print(f'   Columnas: {df_total.shape[1]}')

✅ Guardado: master_total.csv
   Ruta: d:\USER\Desktop\TFM\data\master\master_total.csv
   Filas:    20,316
   Columnas: 122
